In [ ]:
import os
import warnings 
import random 
import copy
import glob 

import pandas as pd 
import sklearn 
import numpy as np

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 
from torchinfo import summary

import torch
from torch import nn 
from torch .utils.data import DataLoader, Dataset 
import torch.nn.functional as F
from torchvision import transforms 
from torchvision.transforms import v2
import cv2
import pydicom

from sklearn.model_selection import GroupKFold

import timm

warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

In [ ]:
SEED = 2026
N_SPLITS = 5
BATCH_SIZE = 64
EPOCHS = 25
LR = 1e-3

def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic=(True)
    torch.backends.cudnn.benchmark=(False)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

In [ ]:
for item in os.listdir(DATA_DIR):
    full_path = os.path.join(DATA_DIR, item)
    if os.path.isdir(full_path):
        print(full_path)

In [ ]:
paths = []
for patient in os.listdir(os.path.join(DATA_DIR, "train_series")):
    files = glob.glob(
        os.path.join(DATA_DIR, "train_series", patient, "**", "*.dcm"),
        recursive=True,
    )
    if files:
        paths.extend(files)
    if len(paths) >= 5:
        break

for path in paths[:5]:
    print(path)

In [ ]:
random_patient = random.choice(
    os.listdir(os.path.join(DATA_DIR, "train_series"))
)
all_dcm_files = glob.glob(
    os.path.join(DATA_DIR, "train_series", random_patient, "**", "*.dcm"),
    recursive=True,
)
print(f"All dcm files in train: {len(all_dcm_files)}")

random_dcm_path = random.choice(all_dcm_files)
dicom_data = pydicom.dcmread(random_dcm_path)
image_array = dicom_data.pixel_array

print(f"Shape image: {image_array.shape}")
print(f"Type data image: {image_array.dtype}")

plt.imshow(image_array, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"Device: '{device}'")

In [ ]:
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

In [ ]:
class RSNAKneeDataset(Dataset):

    def __init__(self, df, img_size=224, is_train=True):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size 
        self.is_train = is_train 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]
        study_path = os.path.join(DATA_DIR, "train_series", study_id)
        dcm_files = sorted(glob.glob(os.path.join(study_path, "**", "*.dcm"), recursive=True))

        if len(dcm_files) > 0:
            mid_idx = len(dcm_files) // 2
            selected_file = dcm_files[mid_idx]

            try:
                dcm_data = pydicom.dcmread(selected_file)
                image = dcm_data.pixel_array.astype(np.float32)

                if image.max() != image.min():
                    image = (image - image.min()) / (image.max() - image.min()) * 255.0 
                image = image.astype(np.uint8)
                image = cv2.resize(image, (self.img_size, self.img_size))
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            except Exception:
                image = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            image = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        image = torch.tensor(image, dtype=torch.float32)

        if self.is_train:
            labels = row[TARGET_COLS].values.astype(np.float32)
            labels = np.nan_to_num(labels, nan=0.0)
            labels = torch.tensor(labels, dtype=torch.float32)
            return image, labels 
        return image 

In [ ]:
class RSNAKneeModel(nn.Module):
    def __init__(self, model_name="efficientnet_b0", pretrained=True):
        super().__init__()

        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=len(TARGET_COLS))

    def forward(self, x):
        return self.model(x)

In [ ]:
from sklearn.metrics import roc_auc_score